In [17]:
import pandas as pd
import torch
train = pd.read_csv("../data/final/dataset_full.csv")
train.head()

,Pricing Date,Issuer Name,Offer Size (M),Offer Price,Offer To 1st Close,Initial Pub Offer (Shares Offered),Industry Sector,Market Cap at Offer (M),Instit Owner (% Shares Out),Primary Exchange,...,has_bulge_bracket,vix,nasdaq,fed_funds,treasury_10y,cpi,unemployment,gdp,ipo_volume,market_return_1m
0,2000-01-24,Neoforma Inc,91.00,13.0,302.884613,7000000.0,Technology,732.744,0.014765,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
1,2000-01-24,Townsquare Media 2010 Inc,136.00,8.5,0.000000,16000000.0,Communications,272.983,NaN,,...,0,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
2,2000-01-25,Healthgate Data Corp,41.25,11.0,6.818182,3750000.0,Communications,180.900,NaN,,...,0,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
3,2000-01-25,T/R Systems Inc,30.00,10.0,59.380001,3000000.0,Technology,115.002,NaN,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
4,2000-01-26,John Hancock Financial Services Inc,1734.00,17.0,3.676471,102000000.0,Financial,5638.900,0.072897,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN


In [5]:
def cross_entropy_loss(Q, Y):
    return -torch.mean(Y * torch.log(Q)) / Y.shape[0]

**Data Prep**

In [6]:
class DataPrepPipeline:
  def __init__(self):
    self.features = ['offer_size_to_mktcap', 'vix']
  def fit(self, X):
    return self
  def transform(self, X):
    eng_features = torch.from_numpy(X[self.features].values).float()
    return eng_features

In [13]:
X_df = train.drop(columns=['underpriced'])
y_df = train['underpriced']
y_df = pd.get_dummies(y_df, columns=['underpriced'])

# 80/20 split from lecture notes 9
train_ix = X_df.sample(frac=0.8, random_state=42).index
test_ix = X_df.drop(train_ix).index

X_train_df = X_df.loc[train_ix]
y_train_df = y_df.loc[train_ix]

X_test_df = X_df.loc[test_ix]
y_test_df = y_df.loc[test_ix]

pipeline = DataPrepPipeline()
pipeline.fit(X_train_df)

X_train = pipeline.transform(X_train_df)
y_train = torch.from_numpy(y_train_df.values).float()

**Model**

In [8]:
class LogisticRegression:
    def __init__(self, d_features, k_classes):
        self.W = torch.zeros(d_features, k_classes)

    def forward(self, X):
        S = X @ self.W
        return torch.softmax(S, dim=1)

**Gradient Descent Optimizer**

In [9]:
class GradientDescentOptimizer:
  def __init__(self, model, learning_rate=0.01):
    self.model = model
    self.learning_rate = learning_rate

  def step(self, X, y):
    self.model.W -= self.learning_rate * self.grad_func(X, y)

  def grad_func(self, X, y):
    q = self.model.forward(X)
    return X.T @ (q - y) / X.shape[0]

**Model Training**

In [14]:
model = LogisticRegression(d_features=X_train.shape[1], k_classes=2)
opt = GradientDescentOptimizer(model, learning_rate=0.01)
losses = []
for epoch in range(50000):
  q = model.forward(X_train)
  loss = cross_entropy_loss(q, y_train)
  losses.append(loss.item())
  opt.step(X_train, y_train)

In [15]:
def acc(X_test_df, y_test_df):
  X_test = pipeline.transform(X_test_df)
  y_test = torch.from_numpy(y_test_df.values).float()
  s_pred = model.forward(X_test)
  y_test_preds = s_pred.argmax(dim=1)
  y_test_labels = y_test.argmax(dim=1).int()
  return (1.0*(y_test_preds == y_test_labels)).mean().item()

In [16]:
print(f"Accuracy: {acc(X_test_df, y_test_df):.2f}")

Accuracy: 0.39
